**Manual vs. Automatic Gradients**
- **Objective:** Understand how PyTorch computes derivatives automatically.
- **Task:** Define a simple multivariate linear regression model formula: y=w_1*x_1+w_2*x_2+b.
- **Action:** 
- 1. Initialize tensors for x_1, x_2 (inputs) and w_1, w_2, b (parameters) setting requires_grad=True on the parameters.
- 2. Perform a forward pass to compute y.
- 3. Compute a simple Mean Squared Error (MSE) loss against a dummy target.
- 4. Call .backward() on the loss.
- 5. Verification: Manually calculate the partial derivatives of the loss with respect to w_1, w_2, and b using math, and compare your manual results to w1.grad, w2.grad, and b.grad.

In [1]:
import torch

# 1. Initialize inputs (no gradients needed for inputs)
x1 = torch.tensor(2.0)
x2 = torch.tensor(3.0)
y_true = torch.tensor(5.0)

# 2. Initialize parameters (requires_grad=True tracks operations for backprop)
w1 = torch.tensor(1.0, requires_grad=True)
w2 = torch.tensor(1.5, requires_grad=True)
b = torch.tensor(0.5, requires_grad=True)

# 3. Forward Pass
y_pred = w1 * x1 + w2 * x2 + b

# 4. Compute Loss (MSE for a single sample)
loss = (y_pred - y_true) ** 2

# 5. Backward Pass (Automatic Differentiation)
loss.backward()

# 6. Verification
print(f"Forward Pass Predicted y: {y_pred.item()} (Expected: 7.0)")
print(f"Computed Loss: {loss.item()} (Expected: 4.0)\n")

print("--- Gradient Comparison ---")
print(f"w1.grad: {w1.grad.item():.1f} | Manual Target: 8.0")
print(f"w2.grad: {w2.grad.item():.1f} | Manual Target: 12.0")
print(f"b.grad:  {b.grad.item():.1f}  | Manual Target: 4.0")

Forward Pass Predicted y: 7.0 (Expected: 7.0)
Computed Loss: 4.0 (Expected: 4.0)

--- Gradient Comparison ---
w1.grad: 8.0 | Manual Target: 8.0
w2.grad: 12.0 | Manual Target: 12.0
b.grad:  4.0  | Manual Target: 4.0


**Custom Autograd Functions**
- **Objective:** Extend Autograd by writing custom forward and backward passes.
- **Task:** Sometimes, you need operations that aren't differentiable by default, or you want to optimize memory. Subclass torch.autograd.Function.
- **Action:**
- 1. Implement a custom activation function (e.g., Swish or a noisy ReLU).
- 2. Write the forward static method, saving necessary context using ctx.save_for_backward().
- 3. Write the backward static method, calculating the chain rule derivative using the saved context and grad_output.
- 4. Apply your custom function to a tensor and verify gradients using torch.autograd.gradcheck.

In [2]:
import torch

class CustomHardSigmoid(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        output = torch.clamp(0.2 * x + 0.5, min=0.0, max=1.0)
        return output

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        grad_input = torch.zeros_like(x)
        mask = (x > -2.5) & (x < 2.5)
        grad_input[mask] = grad_output[mask] * 0.2
        return grad_input

In [3]:
# Initialize data
x = torch.tensor([[-3.0, 0.0, 3.0]], dtype=torch.float64, requires_grad=True)

# Apply the custom function
hard_sigmoid = CustomHardSigmoid.apply
y = hard_sigmoid(x)

print(f"Input:  {x.detach().numpy()}")
print(f"Output: {y.detach().numpy()}")

Input:  [[-3.  0.  3.]]
Output: [[0.  0.5 1. ]]


In [4]:
from torch.autograd import gradcheck

# Verify the gradients match finite differences
test_input = torch.randn(3, 3, dtype=torch.float64, requires_grad=True) * 2.0
test_passed = gradcheck(CustomHardSigmoid.apply, (test_input,), eps=1e-6, atol=1e-4)

print(f"Gradcheck validation successful: {test_passed}")

Gradcheck validation successful: True
